# Task 3 (mở rộng): SDXL thật cho nhiều Storyboard (Kaggle)

Chạy `src.image_generator.story_runs.run_stories`: mỗi Storyboard đã duyệt sinh 4 ảnh SDXL thật và một trang comic 2×2.
Không sinh lại truyện, không retry, không thay ảnh lỗi bằng mock. Truyện lỗi được ghi nguyên nhân rồi chạy tiếp truyện sau.

**Yêu cầu khi chạy**
- Accelerator: GPU T4 (notebook chỉ dùng một GPU, `CUDA_VISIBLE_DEVICES=0`). Internet: bật.
- Storyboard JSON nằm trong Kaggle **private dataset** (không commit vào repo). Sửa `INPUT_DIR` ở Cell 3.
- `DRY_RUN = True` chạy thử bằng `examples/storyboard.json` để kiểm tra tích hợp trước khi có dữ liệu thật.
- Chạy từ đầu (Run All) trong phiên mới. Điền `SOURCE_COMMIT` (40 ký tự) trước khi chạy.
- Giữ notebook không có output khi commit; tải `results.zip` về và lưu ngoài repo.

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # một T4; phải đặt trước khi import torch

import importlib.metadata as md
import platform
import sys

import torch

print("Python:", sys.version)
print("Platform:", platform.platform())
print("torch:", torch.__version__, "| CUDA build:", torch.version.cuda)
assert torch.cuda.is_available(), "Không thấy GPU: bật Accelerator GPU T4 rồi Restart session."
print("GPU:", torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.2f} GiB")
for pkg in ["diffusers", "transformers", "accelerate", "safetensors", "huggingface_hub", "Pillow", "pydantic", "PyYAML"]:
    try:
        print(f"{pkg}: {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg}: KHONG CO")

In [ ]:
import subprocess

REPO_URL = "https://github.com/trannam1084/comic-generation.git"
SOURCE_COMMIT = "<DIEN_SHA_40_KY_TU_SAU_KHI_PUSH>"
WORKDIR = "/kaggle/working/comic-generation"

assert len(SOURCE_COMMIT) == 40 and all(c in "0123456789abcdef" for c in SOURCE_COMMIT), \
    "Điền SOURCE_COMMIT là SHA đầy đủ 40 ký tự (git rev-parse HEAD)."

GIT_ENV = dict(os.environ, GIT_TERMINAL_PROMPT="0")  # không bao giờ chờ nhập mật khẩu


def git(*args):
    done = subprocess.run(["git", *args], capture_output=True, text=True, env=GIT_ENV)
    if done.returncode != 0:
        raise RuntimeError(f"git {args[0]} that bai (ma {done.returncode}): {done.stderr.strip()[:300]}")
    return done.stdout.strip()


if not os.path.isdir(WORKDIR):
    git("clone", "--no-checkout", REPO_URL, WORKDIR)
git("-C", WORKDIR, "checkout", "--detach", SOURCE_COMMIT)
head = git("-C", WORKDIR, "rev-parse", "HEAD")
assert head == SOURCE_COMMIT, f"HEAD {head} khac commit mong doi"
print("HEAD:", head)
print("Thu muc sach (khong co thay doi cuc bo):", git("-C", WORKDIR, "status", "--porcelain") == "")

os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)

In [ ]:
import json
from pathlib import Path

DRY_RUN = True  # True: chạy thử với examples/storyboard.json. Đặt False khi chạy 5 truyện thật.
INPUT_DIR = Path("/kaggle/input/SUA-TEN-DATASET-PRIVATE")  # sửa theo tên dataset private
STORY_IDS = ["story_013", "story_020", "story_025", "story_027", "story_033"]

if DRY_RUN:
    sample = json.loads(Path("examples/storyboard.json").read_text(encoding="utf-8"))
    stories = {"dryrun_1": sample, "dryrun_2": sample}
else:
    missing = [sid for sid in STORY_IDS if not (INPUT_DIR / f"{sid}.json").is_file()]
    if missing:
        raise FileNotFoundError(f"Thiếu file storyboard trong {INPUT_DIR}: {missing}")
    stories = {sid: json.loads((INPUT_DIR / f"{sid}.json").read_text(encoding="utf-8")) for sid in STORY_IDS}

print("DRY_RUN:", DRY_RUN, "| số truyện:", len(stories), "|", list(stories))

In [ ]:
import gc
import logging
import time

from src.configuration import PipelineConfig
from src.image_generator.prompt_builder import PromptBuilder
from src.image_generator.sdxl_generator import SDXLConfig, SDXLImageGenerator
from src.image_generator.story_runs import run_stories
from src.storyboard.compat import LegacyPromptBuilderAdapter

logging.basicConfig(level=logging.INFO, format="%(message)s")

STYLE_E0 = "children's storybook illustration, clean line art, vibrant colors, single scene"
NEG_E0 = ("text, letters, watermark, signature, blurry, low quality, deformed, extra limbs, "
          "multiple panels, collage, split screen, comic page, frame, border, grid")

OUT = Path("/kaggle/working/run_dryrun" if DRY_RUN else "/kaggle/working/run_stories")
if OUT.exists():
    raise RuntimeError("Thư mục kết quả đã tồn tại: xóa hoặc đổi tên trước khi chạy để không ghi đè.")
OUT.mkdir(parents=True)

generator = SDXLImageGenerator(SDXLConfig())  # model nạp ở lần sinh ảnh đầu tiên
base_seed = PipelineConfig().base_seed  # giống Task 3 để so sánh được

run_info = {
    "source_commit": SOURCE_COMMIT,
    "model": {"repo_id": generator.config.repo_id, "revision": generator.config.revision},
    "prompt_style": STYLE_E0,
    "negative_prompt": NEG_E0,
    "accelerator": torch.cuda.get_device_name(0),
    "dry_run": DRY_RUN,
}


def cleanup():
    gc.collect()
    torch.cuda.empty_cache()


start = time.perf_counter()
results = run_stories(
    stories,
    out_root=OUT,
    image_backend=generator,
    base_seed=base_seed,
    prompt_stage=LegacyPromptBuilderAdapter(PromptBuilder(style=STYLE_E0, negative_prompt=NEG_E0)),
    run_info=run_info,
    reset_peak=torch.cuda.reset_peak_memory_stats,
    read_peak_gib=lambda: torch.cuda.max_memory_allocated() / 2**30,
    cleanup=cleanup,
)
total_sec = time.perf_counter() - start

(OUT / "run_info.json").write_text(
    json.dumps(
        {**run_info, "base_seed": base_seed, "model_load_sec": round(generator.model_load_sec, 2),
         "total_sec": round(total_sec, 2)},
        indent=2, ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(f"Nạp model (gồm tải): {generator.model_load_sec:.1f}s | tổng: {total_sec:.1f}s")
for item in results:
    print(item["story_id"], item["status"], item.get("stage"), item.get("pipeline_sec"), item.get("peak_vram_gib"),
          item.get("error_type") or "")

In [ ]:
import hashlib
import shutil


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


archive = Path(shutil.make_archive(str(OUT) + "_zip", "zip", OUT))
print("ZIP:", archive, f"{archive.stat().st_size / 2**20:.1f} MiB", "| sha256:", sha256(archive))
failed = [r["story_id"] for r in results if r["status"] != "success"]
print("Truyện lỗi:", failed or "không có")

In [ ]:
from IPython.display import display
from PIL import Image

for item in results:
    if item["status"] == "success":
        print(item["story_id"])
        display(Image.open(OUT / item["story_id"] / item["comic_path"]).resize((530, 660)))